# Validate regional project progress

## Purpose

Validate `05-analytics.analytics_region_project_progress` independently against Gold. Evidence is persisted before the final STOP gate. The validator accepts a one-snapshot workspace, but records a FLAG and requires every historical measure to remain `NULL`.

This is the validator for proposed D-55. It does not validate or alter the existing eleven Analytics views.

In [0]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `04-validation`;

DECLARE OR REPLACE VARIABLE progress_validation_rule_version STRING
DEFAULT 'regional-project-progress-validation-v1';

CREATE OR REPLACE TEMPORARY VIEW regional_progress_validation_context AS
WITH snapshot_catalog AS (
    SELECT
        source_snapshot_id,
        MAX(snapshot_date_key) AS snapshot_date_key,
        MAX(source_load_ts) AS source_load_ts,
        MAX(run_id) AS fact_run_id,
        MAX(CASE WHEN is_current_snapshot THEN 1 ELSE 0 END) AS is_current_snapshot
    FROM `03-gold`.fact_project_snapshot
    GROUP BY source_snapshot_id
),
current_snapshot AS (
    SELECT source_snapshot_id, snapshot_date_key, fact_run_id
    FROM snapshot_catalog
    WHERE is_current_snapshot = 1
),
previous_snapshot AS (
    SELECT source_snapshot_id, snapshot_date_key, fact_run_id
    FROM snapshot_catalog
    WHERE is_current_snapshot = 0
    QUALIFY ROW_NUMBER() OVER (
        ORDER BY snapshot_date_key DESC, source_load_ts DESC, source_snapshot_id DESC
    ) = 1
),
region_dimension AS (
    SELECT MAX(run_id) AS dim_region_run_id
    FROM `03-gold`.dim_region
)
SELECT
    current.source_snapshot_id AS current_source_snapshot_id,
    current.snapshot_date_key AS current_snapshot_date_key,
    current.fact_run_id AS current_fact_run_id,
    previous.source_snapshot_id AS previous_source_snapshot_id,
    previous.snapshot_date_key AS previous_snapshot_date_key,
    previous.fact_run_id AS previous_fact_run_id,
    region.dim_region_run_id,
    SHA2(CONCAT_WS(
        '|',
        current.fact_run_id,
        COALESCE(previous.fact_run_id, 'NO_PRIOR_SNAPSHOT'),
        region.dim_region_run_id,
        progress_validation_rule_version
    ), 256) AS validation_run_id
FROM current_snapshot AS current
LEFT JOIN previous_snapshot AS previous
    ON TRUE
CROSS JOIN region_dimension AS region;

SELECT ASSERT_TRUE(
    COUNT(*) = 1
    AND COUNT_IF(current_source_snapshot_id IS NULL) = 0,
    'STOP: regional progress validation requires exactly one current non-null DPWH snapshot.'
)
FROM regional_progress_validation_context;

## Upstream gates

**STOP:** Require the current Gold project fact and the existing Analytics views to have passed their validators before checking this extension.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW regional_progress_upstream_gate AS
WITH latest_gold AS (
    SELECT dq.validation_run_id
    FROM `04-validation`.gold_dq_results AS dq
    INNER JOIN regional_progress_validation_context AS context
        ON dq.pipeline_run_id = context.current_fact_run_id
    WHERE dq.table_name = 'fact_project_snapshot'
    QUALIFY ROW_NUMBER() OVER (ORDER BY dq.run_ts DESC, dq.validation_run_id DESC) = 1
),
latest_analytics AS (
    SELECT dq.validation_run_id
    FROM `04-validation`.analytics_dq_results AS dq
    INNER JOIN regional_progress_validation_context AS context
        ON dq.pipeline_run_id = context.current_fact_run_id
        AND dq.source_snapshot_id = context.current_source_snapshot_id
    WHERE dq.rule_version = 'analytics-validation-v2'
        AND dq.table_name = 'analytics_regional_investment'
    QUALIFY ROW_NUMBER() OVER (ORDER BY dq.run_ts DESC, dq.validation_run_id DESC) = 1
)
SELECT
    (
        SELECT COUNT(*)
        FROM `04-validation`.gold_dq_results AS dq
        INNER JOIN latest_gold AS latest
            ON dq.validation_run_id = latest.validation_run_id
        WHERE dq.action = 'stop' AND dq.status IN ('FAIL', 'ERROR')
    ) AS gold_stop_failures,
    (SELECT COUNT(*) FROM latest_gold) AS gold_validation_found,
    (
        SELECT COUNT(*)
        FROM `04-validation`.analytics_dq_results AS dq
        INNER JOIN latest_analytics AS latest
            ON dq.validation_run_id = latest.validation_run_id
        WHERE dq.action = 'stop' AND dq.status IN ('FAIL', 'ERROR')
    ) AS analytics_stop_failures,
    (SELECT COUNT(*) FROM latest_analytics) AS analytics_validation_found;

SELECT ASSERT_TRUE(
    gold_validation_found = 1
    AND gold_stop_failures = 0
    AND analytics_validation_found = 1
    AND analytics_stop_failures = 0,
    'STOP: run passing Gold validation and Analytics v2 validation for the current snapshot before regional progress validation.'
)
FROM regional_progress_upstream_gate;

## Rebuild the contract from Gold

**What:** Calculate independent regional totals and snapshot-transition bounds from Gold.

**Why:** a view must not validate itself. Counts use the stable PSGC region code across snapshots and keep region key `0` outside the published regional grain.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW regional_progress_current_projects AS
SELECT
    fact.project_key,
    region.psgc_region_code,
    status.status_group,
    fact.physical_progress_pct,
    fact.reported_budget_pesos
FROM `03-gold`.fact_project_snapshot AS fact
INNER JOIN `03-gold`.dim_region AS region
    ON fact.region_key = region.region_key
INNER JOIN `03-gold`.dim_project_status AS status
    ON fact.status_key = status.status_key
WHERE fact.is_current_snapshot = TRUE
    AND fact.region_key <> 0;

CREATE OR REPLACE TEMPORARY VIEW regional_progress_previous_projects AS
SELECT
    fact.project_key,
    region.psgc_region_code,
    status.status_group,
    fact.physical_progress_pct,
    fact.reported_budget_pesos
FROM `03-gold`.fact_project_snapshot AS fact
INNER JOIN regional_progress_validation_context AS context
    ON fact.source_snapshot_id = context.previous_source_snapshot_id
INNER JOIN `03-gold`.dim_region AS region
    ON fact.region_key = region.region_key
INNER JOIN `03-gold`.dim_project_status AS status
    ON fact.status_key = status.status_key
WHERE fact.region_key <> 0;

CREATE OR REPLACE TEMPORARY VIEW regional_progress_expected AS
SELECT
    psgc_region_code,
    COUNT(*) AS current_project_count,
    COUNT_IF(status_group = 'FINISHED') AS current_finished_count,
    COUNT_IF(status_group = 'ACTIVE') AS current_active_count,
    COUNT_IF(status_group = 'INACTIVE') AS current_inactive_count,
    COUNT_IF(status_group IS NULL) AS current_unmapped_status_count,
    SUM(reported_budget_pesos) AS current_total_reported_budget_pesos
FROM regional_progress_current_projects
GROUP BY psgc_region_code;

CREATE OR REPLACE TEMPORARY VIEW regional_progress_metrics AS
WITH output AS (
    SELECT * FROM `05-analytics`.analytics_region_project_progress
),
region_set AS (
    SELECT
        COUNT(*) AS output_rows,
        COUNT(DISTINCT region_key) AS output_region_keys,
        COUNT(DISTINCT psgc_region_code) AS output_region_codes,
        COUNT_IF(region_key = 0) AS key_zero_rows,
        COUNT_IF(output.current_source_snapshot_id <> context.current_source_snapshot_id) AS wrong_current_snapshot_rows,
        COUNT_IF(
            context.previous_source_snapshot_id IS NULL
            AND (
                comparison_status <> 'NO_PRIOR_SNAPSHOT'
                OR output.previous_source_snapshot_id IS NOT NULL
                OR output.previous_snapshot_date_key IS NOT NULL
                OR previous_project_count IS NOT NULL
                OR previous_finished_count IS NOT NULL
                OR previous_active_count IS NOT NULL
                OR previous_inactive_count IS NOT NULL
                OR previous_unmapped_status_count IS NOT NULL
                OR previous_total_reported_budget_pesos IS NOT NULL
                OR project_count_change IS NOT NULL
                OR reported_budget_change_pesos IS NOT NULL
                OR status_changed_count IS NOT NULL
                OR moved_to_finished_count IS NOT NULL
                OR progress_increased_count IS NOT NULL
                OR progress_decreased_count IS NOT NULL
                OR new_project_count IS NOT NULL
                OR no_longer_present_count IS NOT NULL
                OR region_changed_count IS NOT NULL
            )
        ) AS unsafe_no_prior_rows,
        COUNT_IF(
            context.previous_source_snapshot_id IS NOT NULL
            AND (
                comparison_status <> 'COMPARABLE'
                OR NOT (output.previous_source_snapshot_id <=> context.previous_source_snapshot_id)
                OR NOT (output.previous_snapshot_date_key <=> context.previous_snapshot_date_key)
            )
        ) AS wrong_previous_snapshot_rows,
        COUNT_IF(
            current_finished_count + current_active_count + current_inactive_count
                + current_unmapped_status_count <> current_project_count
        ) AS current_status_sum_breaks,
        COUNT_IF(
            current_project_count > 0
            AND ABS(
                COALESCE(current_finished_pct, 0) + COALESCE(current_active_pct, 0)
                + COALESCE(current_inactive_pct, 0) + COALESCE(current_unmapped_status_pct, 0) - 100
            ) > 0.04
        ) AS current_pct_breaks,
        COUNT_IF(
            context.previous_source_snapshot_id IS NOT NULL
            AND (
                status_changed_count > current_project_count + no_longer_present_count
                OR moved_to_finished_count > status_changed_count
                OR progress_increased_count > current_project_count + no_longer_present_count
                OR progress_decreased_count > current_project_count + no_longer_present_count
                OR region_changed_count > current_project_count + no_longer_present_count
            )
        ) AS impossible_transition_rows,
        COALESCE(SUM(current_project_count), 0) AS output_current_projects,
        COALESCE(SUM(current_total_reported_budget_pesos), 0) AS output_current_budget,
        COALESCE(SUM(progress_decreased_count), 0) AS progress_decreased_projects
    FROM output
    CROSS JOIN regional_progress_validation_context AS context
),
expected_set AS (
    SELECT
        COUNT(*) AS expected_region_rows,
        COALESCE(SUM(current_project_count), 0) AS expected_current_projects,
        COALESCE(SUM(current_total_reported_budget_pesos), 0) AS expected_current_budget
    FROM regional_progress_expected
),
row_compare AS (
    SELECT COUNT_IF(
        expected.psgc_region_code IS NULL
        OR output.psgc_region_code IS NULL
        OR NOT (output.current_project_count <=> COALESCE(expected.current_project_count, 0))
        OR NOT (output.current_finished_count <=> COALESCE(expected.current_finished_count, 0))
        OR NOT (output.current_active_count <=> COALESCE(expected.current_active_count, 0))
        OR NOT (output.current_inactive_count <=> COALESCE(expected.current_inactive_count, 0))
        OR NOT (output.current_unmapped_status_count <=> COALESCE(expected.current_unmapped_status_count, 0))
        OR NOT (output.current_total_reported_budget_pesos <=> expected.current_total_reported_budget_pesos)
    ) AS current_measure_breaks
    FROM `05-analytics`.analytics_region_project_progress AS output
    FULL OUTER JOIN regional_progress_expected AS expected
        ON output.psgc_region_code = expected.psgc_region_code
    WHERE output.current_project_count > 0 OR expected.psgc_region_code IS NOT NULL
),
official_regions AS (
    SELECT COUNT(*) AS official_region_rows
    FROM `05-analytics`.analytics_regional_investment
)
SELECT *
FROM region_set
CROSS JOIN expected_set
CROSS JOIN row_compare
CROSS JOIN official_regions;

SELECT * FROM regional_progress_metrics;

## Persist evidence, then enforce STOP checks

**STOP:** grain, region set, current measures, lineage, no-prior behavior, and transition bounds must be safe.

**FLAG:** a missing prior snapshot means change measures are unavailable, not zero. Decreased progress is retained for review because a later source can correct an earlier value.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW regional_progress_validation_checks AS
WITH context AS (SELECT * FROM regional_progress_validation_context),
metrics AS (SELECT * FROM regional_progress_metrics),
checks AS (
    SELECT 'Uniqueness' AS attribute, 'One row per official region' AS check_name,
        output_rows - output_region_keys AS failed_rows, output_rows AS total_rows, 'stop' AS action,
        'Grain: region_key' AS details FROM metrics
    UNION ALL
    SELECT 'Uniqueness', 'One row per stable PSGC region code', output_rows - output_region_codes, output_rows, 'stop',
        'Comparison grain: psgc_region_code' FROM metrics
    UNION ALL
    SELECT 'Completeness', 'Every current official region stays visible',
        ABS(output_rows - official_region_rows), official_region_rows, 'stop',
        'Rows must match analytics_regional_investment' FROM metrics
    UNION ALL
    SELECT 'Validity', 'Region key 0 is excluded', key_zero_rows, output_rows, 'stop',
        'Unknown and unresolved geography is reviewed outside this regional view' FROM metrics
    UNION ALL
    SELECT 'Accuracy', 'Current regional measures rebuild from Gold', current_measure_breaks, output_rows, 'stop',
        'Counts, status groups, and reported budget' FROM metrics
    UNION ALL
    SELECT 'Consistency', 'Current mapped project total reconciles to Gold',
        ABS(output_current_projects - expected_current_projects), expected_current_projects, 'stop',
        'Current Gold rows with region_key other than 0' FROM metrics
    UNION ALL
    SELECT 'Consistency', 'Current mapped reported budget reconciles to Gold',
        CAST(ABS(output_current_budget - expected_current_budget) > 0.01 AS BIGINT), expected_region_rows, 'stop',
        'Reported budget is not payment' FROM metrics
    UNION ALL
    SELECT 'Auditability', 'Current snapshot lineage is exact', wrong_current_snapshot_rows, output_rows, 'stop',
        'Every row carries the current Gold source snapshot' FROM metrics
    UNION ALL
    SELECT 'Validity', 'Current status counts and percentages reconcile',
        current_status_sum_breaks + current_pct_breaks, output_rows, 'stop',
        'FINISHED, ACTIVE, INACTIVE, and unmapped status' FROM metrics
    UNION ALL
    SELECT 'Consistency', 'No-prior workspace cannot publish historical zeros', unsafe_no_prior_rows, output_rows, 'stop',
        'Historical fields must be NULL with NO_PRIOR_SNAPSHOT' FROM metrics
    UNION ALL
    SELECT 'Auditability', 'Previous snapshot lineage is exact when available', wrong_previous_snapshot_rows, output_rows, 'stop',
        'Uses the immediately preceding Gold snapshot' FROM metrics
    UNION ALL
    SELECT 'Validity', 'Transition counts stay within project bounds', impossible_transition_rows, output_rows, 'stop',
        'Moved-to-finished is a subset of status changes' FROM metrics
    UNION ALL
    SELECT 'Timeliness', 'A previous snapshot is available for comparison',
        CAST(context.previous_source_snapshot_id IS NULL AS BIGINT), 1, 'flag',
        'NO_PRIOR_SNAPSHOT means current status works but change measures are unavailable'
    FROM metrics CROSS JOIN context
    UNION ALL
    SELECT 'Consistency', 'Projects with lower reported physical progress need review', progress_decreased_projects,
        output_current_projects, 'flag',
        'A decrease can be a source correction and is not automatically an error' FROM metrics
)
SELECT
    context.validation_run_id,
    context.current_fact_run_id AS pipeline_run_id,
    'analytics' AS layer,
    'analytics_region_project_progress' AS table_name,
    context.current_source_snapshot_id AS source_snapshot_id,
    progress_validation_rule_version AS rule_version,
    checks.attribute AS data_quality_attribute,
    checks.check_name AS data_quality_check,
    CAST(checks.failed_rows AS BIGINT) AS failed_rows,
    CAST(checks.total_rows AS BIGINT) AS total_rows,
    ROUND(100.0 * checks.failed_rows / NULLIF(checks.total_rows, 0), 4) AS percentage,
    CASE
        WHEN checks.action = 'stop' AND checks.failed_rows > 0 THEN 'FAIL'
        WHEN checks.action = 'flag' AND checks.failed_rows > 0 THEN 'FLAG'
        ELSE 'PASS'
    END AS status,
    checks.action,
    checks.details,
    CURRENT_TIMESTAMP() AS run_ts
FROM checks
CROSS JOIN context;

CREATE TABLE IF NOT EXISTS `04-validation`.analytics_dq_results (
    validation_run_id STRING NOT NULL, pipeline_run_id STRING NOT NULL, layer STRING NOT NULL,
    table_name STRING NOT NULL, source_snapshot_id STRING, rule_version STRING,
    data_quality_attribute STRING NOT NULL, data_quality_check STRING NOT NULL,
    failed_rows BIGINT, total_rows BIGINT, percentage DOUBLE, status STRING NOT NULL,
    action STRING NOT NULL, details STRING, run_ts TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Analytics validation evidence. One row per check per validation run.';

MERGE INTO `04-validation`.analytics_dq_results AS target
USING regional_progress_validation_checks AS source
ON target.validation_run_id = source.validation_run_id
AND target.table_name = source.table_name
AND target.data_quality_check = source.data_quality_check
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(status = 'FAIL' AND action = 'stop') = 0,
    'STOP: regional project progress validation failed. Review analytics_dq_results.'
)
FROM `04-validation`.analytics_dq_results
WHERE validation_run_id = (SELECT validation_run_id FROM regional_progress_validation_context)
    AND table_name = 'analytics_region_project_progress';

## Summary

The results below are the evidence for this deterministic validation run. A FLAG does not block publication. A failed STOP remains saved even though the final gate stops the notebook.

In [0]:
%sql
SELECT
    status,
    action,
    failed_rows,
    total_rows,
    percentage,
    data_quality_check,
    details
FROM `04-validation`.analytics_dq_results
WHERE validation_run_id = (SELECT validation_run_id FROM regional_progress_validation_context)
    AND table_name = 'analytics_region_project_progress'
ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'FLAG' THEN 2 ELSE 3 END, data_quality_check;